In [0]:
# ================================================================
# NOTEBOOK: nb_gold_product_performance
# PURPOSE:  Product-level performance analytics
# RUN:      Daily (after all silver notebooks complete)
# READS:
#   silver/orderitems/
#   silver/products/
#   silver/productreviews/
#   silver/returns/
#   silver/orders/
# WRITES:   gold/product_performance/
# GRAIN:    1 row per ProductID (all-time summary)
#
#   IMPORTANT RETURN LIMITATION:
#   Returns contains OrderID but not OrderItemID/ProductID.
#   Therefore, exact product returns cannot be calculated.
#   This notebook uses proxy metrics:
#     ReturnedProductOrders
#     UnitsInReturnedOrders
#     ReturnedOrderRate
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, avg, coalesce

STORAGE = "abfss://source@stshopsensedevhj.dfs.core.windows.net"
GOLD_PATH = f"{STORAGE}/gold/product_performance/"

items = spark.read.format("delta").load(f"{STORAGE}/silver/orderitems/")
products = spark.read.format("delta").load(f"{STORAGE}/silver/products/")
returns = spark.read.format("delta").load(f"{STORAGE}/silver/returns/")
reviews = spark.read.format("delta").load(f"{STORAGE}/silver/productreviews/")
orders = spark.read.format("delta").load(f"{STORAGE}/silver/orders/")


# ================================================================
# STEP 1: KEEP ONLY ACTIVE SILVER RECORDS
# (This is the only filtering step - a duplicate STEP 2 that
#  overwrote this with broken logic has been removed.)
# ================================================================

active_items = items.filter(
    F.coalesce(col("_is_deleted"), lit(False)) == False
)

active_orders = orders.filter(
    F.coalesce(col("_is_deleted"), lit(False)) == False
)

active_returns = returns.filter(
    F.coalesce(col("_is_deleted"), lit(False)) == False
)

print(
    f"""
[ACTIVE RECORDS]
Order items : {active_items.count()}
Orders      : {active_orders.count()}
Returns     : {active_returns.count()}
Products    : {products.count()}
Reviews     : {reviews.count()}
"""
)


# ================================================================
# STEP 2: VALIDATE PRODUCT GRAIN
# ================================================================

null_product_ids = products.filter(col("ProductID").isNull()).count()

duplicate_products = (
    products.groupBy("ProductID").count().filter(col("count") > 1)
)
duplicate_product_count = duplicate_products.count()

print(
    f"""
[PRODUCT VALIDATION]
Null ProductIDs      : {null_product_ids}
Duplicate ProductIDs : {duplicate_product_count}
"""
)

assert null_product_ids == 0, "Products table contains null ProductID values."
assert duplicate_product_count == 0, "Products table contains duplicate ProductID values."


# ================================================================
# STEP 3: ADD ORDER CONTEXT TO ORDER ITEMS
# ================================================================

order_context = active_orders.select(
    "OrderID", "CustomerID", "OrderDate", "OrderYear", "OrderMonth",
    "IsDelivered", "IsCancelled", "IsReturned"
)

items_with_orders = active_items.join(order_context, on="OrderID", how="left")

print(f"[JOIN] Items with order context: {items_with_orders.count()} rows")


# ================================================================
# STEP 4: SALES METRICS PER PRODUCT (GRAIN: 1 row per ProductID)
# ================================================================

sales_metrics = (
    items_with_orders
    .filter(col("IsDelivered") == True)
    .groupBy("ProductID")
    .agg(
        F.round(F.sum("TotalPrice"), 2).alias("GrossRevenue"),
        F.round(F.sum("DiscountAmount"), 2).alias("TotalDiscounts"),
        F.round(F.sum("NetPrice"), 2).alias("NetRevenue"),
        F.sum("Quantity").alias("TotalUnitsSold"),
        F.count("OrderItemID").alias("TotalLineItems"),
        # Fixed: was countDistinct("OrderItemID"), which just re-counts
        # line items since OrderItemID is unique per row.
        F.countDistinct("OrderID").alias("TotalOrders"),
        # Fixed: was countDistinct("OrderID") - that's orders, not customers.
        F.countDistinct("CustomerID").alias("UniqueCustomers"),
        F.round(avg("DiscountPct"), 2).alias("AvgDiscountPct"),
        F.sum(F.when(col("IsDiscounted") == True, 1).otherwise(0)).alias("DiscountedLineItems"),
        F.min("OrderDate").alias("FirstSaleDate"),
        F.max("OrderDate").alias("LastSaleDate")
    )
)
print(f"[SALES METRICS] {sales_metrics.count()} products have delivered sales")


# ================================================================
# STEP 5: REVIEW METRICS PER PRODUCT (GRAIN: 1 row per ProductID)
# ================================================================

review_metrics = (
    reviews
    .filter(col("ProductID").isNotNull())
    .groupBy("ProductID")
    .agg(
        # Fixed: was countDistinct("ProductID"), which is always 1
        # per group since ProductID is the grouping key.
        F.count("ProductID").alias("TotalReviews"),
        F.round(avg("Rating"), 2).alias("AvgRating"),
        F.sum(F.when(col("SentimentFlag") == "POSITIVE", 1).otherwise(0)).alias("PositiveReviews"),
        F.sum(F.when(col("SentimentFlag") == "NEUTRAL", 1).otherwise(0)).alias("NeutralReviews"),
        F.sum(F.when(col("SentimentFlag") == "NEGATIVE", 1).otherwise(0)).alias("NegativeReviews"),
        F.sum(F.when(col("IsVerifiedBool") == True, 1).otherwise(0)).alias("VerifiedReviews"),
        F.sum(F.when(col("IsHighQualityReview") == True, 1).otherwise(0)).alias("HighQualityReviews"),
        F.sum(F.when(col("IsSuspicious") == True, 1).otherwise(0)).alias("SuspiciousReviews"),

        F.sum(F.when(col("HelpfulnessScore") >= 0.7, 1).otherwise(0)).alias("HighHelpfulnessReviews"),
        F.round(F.avg("HelpfulnessScore"), 2).alias("AvgHelpfulness"),
        F.max("ReviewDate").alias("LatestReviewDate")
    )
)
print(f"[REVIEW METRICS] {review_metrics.count()} products have reviews")


# ================================================================
# STEP 6: CREATE RETURNED ORDER LIST
# ================================================================

returned_orders = (
    active_returns
    .filter(col("OrderID").isNotNull())
    .select("OrderID").distinct()
    .withColumn("HasReturnRecord", lit(True))
)
print(f"[RETURNED ORDERS] {returned_orders.count()} distinct orders have return records")


# ================================================================
# STEP 7: RETURNED-ORDER PROXY METRICS PER PRODUCT
# ================================================================

items_in_returned_orders = active_items.join(returned_orders, on="OrderID", how="inner")

returned_order_metrics = (
    items_in_returned_orders
    .groupBy("ProductID")
    .agg(
        F.countDistinct("OrderID").alias("ReturnedProductOrders"),
        F.sum("Quantity").alias("UnitsInReturnedOrders")
    )
)
print(f"[RETURN PROXY METRICS] {returned_order_metrics.count()} products appeared in returned orders")


# ================================================================
# STEP 8: JOIN ALL METRICS WITH PRODUCT DIMENSION
# ================================================================

gold_df = (
    products
    .join(sales_metrics, on="ProductID", how="left")
    .join(review_metrics, on="ProductID", how="left")
    .join(returned_order_metrics, on="ProductID", how="left")
)
print(f"[GOLD JOIN] Rows before derived metrics: {gold_df.count()}")


# ================================================================
# STEP 9: FILL NULL METRIC VALUES
# ================================================================

gold_df = gold_df.fillna({
    "GrossRevenue": 0.0, "TotalDiscounts": 0.0, "NetRevenue": 0.0,
    "TotalUnitsSold": 0, "TotalLineItems": 0, "TotalOrders": 0,
    "UniqueCustomers": 0, "AvgDiscountPct": 0.0, "DiscountedLineItems": 0,
    "TotalReviews": 0, "AvgRating": 0.0, "PositiveReviews": 0,
    "NeutralReviews": 0, "NegativeReviews": 0, "VerifiedReviews": 0,
    "HighQualityReviews": 0, "HighHelpfulnessReviews": 0, "SuspiciousReviews": 0,
    "AvgHelpfulness": 0.0,
    "ReturnedProductOrders": 0, "UnitsInReturnedOrders": 0
})


# ================================================================
# STEP 10: DERIVED BUSINESS METRICS
# ================================================================

gold_df = (
    gold_df
    .withColumn(
        "ReturnedOrderRate",
        F.when(col("TotalOrders") > 0,
               F.round(col("ReturnedProductOrders") / col("TotalOrders") * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "ReviewCoverage",
        F.when(col("TotalOrders") > 0,
               F.round(col("TotalReviews") / col("TotalOrders") * 100, 2)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "PositiveReviewRate",
        when(col("TotalReviews") > 0,
             F.round(col("PositiveReviews") / col("TotalReviews") * 100, 2)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "VerifiedReviewRate",
        when(col("TotalReviews") > 0,
             F.round(col("VerifiedReviews") / col("TotalReviews") * 100, 2)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "RevenuePerUnit",
        when(col("TotalUnitsSold") > 0,
             F.round(col("NetRevenue") / col("TotalUnitsSold"), 2)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "AvgRevenuePerOrder",
        when(col("TotalOrders") > 0,
             F.round(col("NetRevenue") / col("TotalOrders"), 2)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "EstimatedCost",
        F.round(col("TotalUnitsSold") * F.coalesce(col("CostPrice"), lit(0.0)), 2)
    )
    .withColumn(
        "EstimatedProfit",
        F.round(col("NetRevenue") - col("EstimatedCost"), 2)
    )
    .withColumn(
        "EstimatedProfitMarginPct",
        F.when(col("NetRevenue") > 0,
               F.round(col("EstimatedProfit") / col("NetRevenue") * 100, 2)
        ).otherwise(lit(0.0))
    )
    .withColumn("HasSales", col("TotalOrders") > 0)
    .withColumn("HasReviews", col("TotalReviews") > 0)
    .withColumn("AppearedInReturnedOrder", col("ReturnedProductOrders") > 0)
    .withColumn("ReturnMetricType", lit("ORDER_LEVEL_PROXY"))
)


# ================================================================
# STEP 11: PRODUCT HEALTH SCORE (0-100)
# ================================================================

gold_df = (
    gold_df
    .withColumn(
        "HealthScore",
        when(col("TotalOrders") == 0, lit(0.0))
        .otherwise(
            F.round(
                (col("AvgRating") / 5.0 * 45)
                + F.greatest(lit(0.0), lit(15.0) - (col("ReturnedOrderRate") * 1.5))
                + (col("PositiveReviewRate") / 100.0 * 40),
                2
            )
        )
    )
)


# ================================================================
# STEP 12: PRODUCT CLASSIFICATION
# ================================================================

gold_df = (
    gold_df
    .withColumn(
        "ProductLabel",
        F.when(col("TotalOrders") == 0, "NO_SALES")
        .when(
            (col("NetRevenue") >= 100000) & (col("AvgRating") >= 4.0),
            "STAR"
        )
        .when(
            (col("NetRevenue") >= 100000) & (col("AvgRating") < 4.0),
            "CASH_COW"
        )
        .when(
            (col("NetRevenue") < 100000)
            & (col("AvgRating") >= 4.0)
            & (col("TotalReviews") >= 5),
            "HIDDEN_GEM"
        )
        .when(
            (col("NetRevenue") < 100000)
            & (col("AvgRating") < 3.0)
            & (col("TotalReviews") > 0),
            "UNDERPERFORMER"
        )
        .otherwise("AVERAGE")
    )
    .withColumn("_gold_load_ts", F.current_timestamp())
)


# ================================================================
# STEP 13: SELECT FINAL GOLD COLUMNS
# ================================================================

gold_df = gold_df.select(
    # PRODUCT DIMENSION
    "ProductID", "ProductName", "Category", "SubCategory", "Brand",
    "SellerID", "ListPrice", "CostPrice", "StockQuantity", "IsActiveBool",
    "LaunchDate", "PriceBand", "StockStatus", "IsInStock", "DaysSinceLaunch",
    "IsNewProduct", "RatingBand", "IsHighPotential",
    # SALES METRICS
    "GrossRevenue", "TotalDiscounts", "NetRevenue", "TotalUnitsSold",
    "TotalLineItems", "TotalOrders", "UniqueCustomers", "AvgDiscountPct",
    "DiscountedLineItems", "FirstSaleDate", "LastSaleDate",
    # REVIEW METRICS
    "TotalReviews", "AvgRating", "PositiveReviews", "NeutralReviews",
    "NegativeReviews", "VerifiedReviews", "HighQualityReviews",
    "HighHelpfulnessReviews", "SuspiciousReviews", "AvgHelpfulness", "LatestReviewDate",
    # RETURNED-ORDER PROXY METRICS
    "ReturnedProductOrders", "UnitsInReturnedOrders", "ReturnedOrderRate",
    "AppearedInReturnedOrder", "ReturnMetricType",
    # DERIVED BUSINESS METRICS
    "ReviewCoverage", "PositiveReviewRate", "VerifiedReviewRate",
    "RevenuePerUnit", "AvgRevenuePerOrder", "EstimatedCost", "EstimatedProfit",
    "EstimatedProfitMarginPct", "HealthScore", "ProductLabel",
    "HasSales", "HasReviews",
    # GOLD METADATA
    "_gold_load_ts"
)


# ================================================================
# STEP 14: FINAL DATA QUALITY VALIDATION
# ================================================================

gold_row_count = gold_df.count()
distinct_gold_products = gold_df.select("ProductID").distinct().count()
source_product_count = products.select("ProductID").distinct().count()
null_gold_product_ids = gold_df.filter(col("ProductID").isNull()).count()
negative_orders = gold_df.filter(col("TotalOrders") < 0).count()

# Fixed: was missing entirely, but was referenced below
duplicate_gold_products = (
    gold_df.groupBy("ProductID").count().filter(col("count") > 1).count()
)

# Fixed: was col(("ReturnedOrderRate") < 0) - comparing a string to an
# int outside col(), and an unquoted undefined name for the second check
invalid_return_rates = (
    gold_df
    .filter((col("ReturnedOrderRate") < 0) | (col("ReturnedOrderRate") > 100))
    .count()
)

print(
    f"""
[FINAL GOLD VALIDATION]
Source distinct products : {source_product_count}
Gold rows                : {gold_row_count}
Gold distinct products   : {distinct_gold_products}
Null ProductIDs          : {null_gold_product_ids}
Duplicate ProductIDs     : {duplicate_gold_products}
Negative TotalOrders     : {negative_orders}
Invalid return rates     : {invalid_return_rates}
"""
)

assert gold_row_count == source_product_count, "Gold row count does not match distinct source products."
assert gold_row_count == distinct_gold_products, "Gold table does not have one row per ProductID."
assert duplicate_gold_products == 0, "Gold table contains duplicate ProductID values."
assert negative_orders == 0, "Gold table contains negative TotalOrders."
assert invalid_return_rates == 0, "ReturnedOrderRate contains values outside 0-100."


# ================================================================
# STEP 15: WRITE GOLD DELTA TABLE
# ================================================================

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)

print(f"\n[DONE]\nGold table: {GOLD_PATH}\nRows written: {gold_row_count}")


# ================================================================
# STEP 16: READ BACK AND VERIFY
# ================================================================

gold_check = spark.read.format("delta").load(GOLD_PATH)
print(f"[READ BACK CHECK] {gold_check.count()} rows found in Gold")


# ================================================================
# STEP 17: TOP 10 PRODUCTS BY REVENUE
# ================================================================

print("\n[TOP 10 PRODUCTS BY REVENUE]")
(
    gold_df
    .filter(col("HasSales") == True)
    .select(
        "ProductID", "ProductName", "Category", "Brand", "NetRevenue",
        "TotalUnitsSold", "TotalOrders", "AvgRating", "ReturnedProductOrders",
        "ReturnedOrderRate", "EstimatedProfit", "ProductLabel", "HealthScore"
    )
    .orderBy(col("NetRevenue").desc())
    .limit(10)
    .show(truncate=False)
)


# ================================================================
# STEP 18: HIDDEN GEMS
# ================================================================

print("\n[HIDDEN GEMS]")
(
    gold_df
    .filter(col("ProductLabel") == "HIDDEN_GEM")
    .select(
        "ProductID", "ProductName", "Category", "Brand", "NetRevenue",
        "TotalUnitsSold", "AvgRating", "TotalReviews", "PositiveReviewRate", "HealthScore"
    )
    .orderBy(col("AvgRating").desc(), col("TotalReviews").desc())
    .limit(10)
    .show(truncate=False)
)


# ================================================================
# STEP 19: PRODUCTS FREQUENTLY APPEARING IN RETURNED ORDERS
# (Do not describe these as exact returned products - see notice at top)
# ================================================================

print("\n[PRODUCTS ASSOCIATED WITH RETURNED ORDERS]")
(
    gold_df
    .filter(col("ReturnedProductOrders") > 0)
    .select(
        "ProductID", "ProductName", "Category", "TotalOrders",
        "ReturnedProductOrders", "ReturnedOrderRate", "NetRevenue",
        "AvgRating", "ProductLabel"
    )
    .orderBy(col("ReturnedOrderRate").desc(), col("ReturnedProductOrders").desc())
    .limit(10)
    .show(truncate=False)
)


# ================================================================
# STEP 20: FINAL DISPLAY FOR DATABRICKS
# ================================================================

display(
    gold_df
    .select(
        "ProductID", "ProductName", "Category", "Brand", "PriceBand",
        "NetRevenue", "TotalUnitsSold", "TotalOrders", "AvgRating",
        "TotalReviews", "ReturnedProductOrders", "ReturnedOrderRate",
        "PositiveReviewRate", "EstimatedProfit", "HealthScore",
        "ProductLabel", "ReturnMetricType"
    )
    .orderBy(col("NetRevenue").desc())
    .limit(30)
)


[ACTIVE RECORDS]
Order items : 5945
Orders      : 3015
Returns     : 409
Products    : 300
Reviews     : 1500


[PRODUCT VALIDATION]
Null ProductIDs      : 0
Duplicate ProductIDs : 0

[JOIN] Items with order context: 5945 rows
[SALES METRICS] 299 products have delivered sales
[REVIEW METRICS] 299 products have reviews
[RETURNED ORDERS] 407 distinct orders have return records
[RETURN PROXY METRICS] 282 products appeared in returned orders
[GOLD JOIN] Rows before derived metrics: 300

[FINAL GOLD VALIDATION]
Source distinct products : 300
Gold rows                : 300
Gold distinct products   : 300
Null ProductIDs          : 0
Duplicate ProductIDs     : 0
Negative TotalOrders     : 0
Invalid return rates     : 0


[DONE]
Gold table: abfss://source@stshopsensedevhj.dfs.core.windows.net/gold/product_performance/
Rows written: 300
[READ BACK CHECK] 300 rows found in Gold

[TOP 10 PRODUCTS BY REVENUE]
+---------+-------------------------------+-----------+--------+----------+--------------

ProductID,ProductName,Category,Brand,PriceBand,NetRevenue,TotalUnitsSold,TotalOrders,AvgRating,TotalReviews,ReturnedProductOrders,ReturnedOrderRate,PositiveReviewRate,EstimatedProfit,HealthScore,ProductLabel,ReturnMetricType
PROD0179,Lakme Homekitchen Product 179,Homekitchen,Nike,LUXURY,427911.57,50,17,3.75,8,2,11.76,37.5,17914.07,48.75,CASH_COW,ORDER_LEVEL_PROXY
PROD0099,Prestige Clothing Product 99,Clothing,Nike,LUXURY,379472.75,39,17,3.63,8,7,41.18,25.0,-182361.25,42.67,CASH_COW,ORDER_LEVEL_PROXY
PROD0240,Lakme Beauty Product 240,Beauty,Lakme,BUDGET,371977.29,51,15,3.82,11,5,33.33,45.45,358292.46,52.56,CASH_COW,ORDER_LEVEL_PROXY
PROD0010,Apple Electronics Product 10,Electronics,Bosch,PREMIUM,362864.56,47,19,3.56,9,5,26.32,11.11,239039.77,36.48,CASH_COW,ORDER_LEVEL_PROXY
PROD0174,Adidas Homekitchen Product 174,Homekitchen,Prestige,LUXURY,361173.37,42,14,4.29,7,3,21.43,28.57,-183503.63,50.04,STAR,ORDER_LEVEL_PROXY
PROD0274,Lakme Sports Product 274,Sports,Samsung,LUXURY,352529.60,50,17,3.86,7,8,47.06,28.57,-162105.4,46.17,CASH_COW,ORDER_LEVEL_PROXY
PROD0229,Lakme Beauty Product 229,Beauty,Apple,PREMIUM,338275.82,49,17,3.5,2,4,23.53,0.0,27039.09,31.5,CASH_COW,ORDER_LEVEL_PROXY
PROD0300,Lakme Sports Product 300,Sports,Bosch,PREMIUM,328348.33,35,13,3.86,7,2,15.38,42.86,69713.38,51.88,CASH_COW,ORDER_LEVEL_PROXY
PROD0013,Bosch Electronics Product 13,Electronics,Apple,PREMIUM,318955.17,35,14,3.67,3,4,28.57,33.33,269757.07,46.36,CASH_COW,ORDER_LEVEL_PROXY
PROD0192,Samsung Homekitchen Product 192,Homekitchen,Nike,BUDGET,315544.86,45,18,3.75,4,6,33.33,25.0,310913.91,43.75,CASH_COW,ORDER_LEVEL_PROXY


In [0]:
print("\n[ORDER ITEMS SCHEMA]")
items.printSchema()

print("\n[PRODUCTS SCHEMA]")
products.printSchema()

print("\n[REVIEWS SCHEMA]")
reviews.printSchema()

print("\n[RETURNS SCHEMA]")
returns.printSchema()

print("\n[ORDERS SCHEMA]")
orders.printSchema()

In [0]:
reviews.printSchema()